Copyright 2026 Google LLC

Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at

    https://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/google-research/google-research/blob/master/betum_tool/models/yolo26/notebooks/template_train_and_infer.ipynb)

# Group 2 — YOLO26: Supervised Fruit Maturity Detection & Yield Estimation

This notebook guides you through the end-to-end pipeline to train, validate, and evaluate **YOLO26** on the Coffee & Cashew maturity dataset. 

## Strategic Alignment
* **Core Task:** Supervised object detection trained on close-up crop imagery.
* **Evaluation Metric:** Unified COCOEval AP@50 (Average Precision at 50% IoU).
* **Aesthetic standard:** Professional structure, clean visualization, and interactive threshold sweeps.

## 1. Setup & Dependencies

In [ ]:
# @title 1a. Install dependencies
!pip install -q ultralytics pycocotools matplotlib Pillow requests

In [ ]:
# @title 1b. Clone the repo (for local testing) [TO BE DEPRECATED]
import os

REPO_DIR = "google-research/betum_tool"

if not os.path.exists("google-research"):
    repo_url = "https://github.com/google-research/google-research.git"
    print("Cloning google-research monorepo (sparse checkout)...")
    # We do a sparse checkout of only the betum_tool directory to avoid downloading 2GB+ of monorepo code.
    !git clone --depth=1 --no-checkout {repo_url} google-research
    !cd google-research && git sparse-checkout set betum_tool && git checkout
    print("✓ Cloned google-research/betum_tool/")
else:
    print("Repo already cloned.")

# Verify common scripts exist
assert os.path.exists(f"{REPO_DIR}/common/yolo_to_coco.py"), "Missing yolo_to_coco.py"
assert os.path.exists(f"{REPO_DIR}/common/class_map.json"), "Missing class_map.json"
print("✓ Common scripts found and updated.")

In [ ]:
# @title 1c. Download dataset from Mendeley
import requests
import subprocess

DATA_DIR = "data"
MENDELEY_URL = "https://data.mendeley.com/public-api/zip/r46c6bpfpf/download/1"
ZIP_NAME = "dataset.zip"

os.makedirs(DATA_DIR, exist_ok=True)
zip_path = os.path.join(DATA_DIR, ZIP_NAME)

if not os.path.exists(zip_path):
    print("Downloading dataset from Mendeley (this may take a few minutes)...")
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
    }
    try:
        response = requests.get(MENDELEY_URL, headers=headers, stream=True)
        if response.status_code == 200:
            with open(zip_path, 'wb') as f:
                for chunk in response.iter_content(chunk_size=8192):
                    f.write(chunk)
            print(f"Downloaded to {zip_path}")
        else:
            raise Exception(f"HTTP Error {response.status_code}")
    except Exception as e:
        print(f"Requests failed: {e}. Trying wget fallback...")
        subprocess.run(["wget", "-U", "Mozilla/5.0", "-O", zip_path, MENDELEY_URL], check=True)
else:
    print(f"Dataset already downloaded at {zip_path}")

In [ ]:
# @title 1d. Unzip main archive
import zipfile

print("Unzipping main archive...")
with zipfile.ZipFile(zip_path, 'r') as z:
    z.extractall(DATA_DIR)
print("Done!")

In [ ]:
# @title 1e. Extract .rar archives
print("Searching for .rar files...")
rar_files = []
for root, dirs, files in os.walk(DATA_DIR):
    for file in files:
        if file.endswith(".rar"):
            rar_files.append(os.path.join(root, file))

print(f"Found rar files: {rar_files}")
for rar_path in rar_files:
    print(f"Extracting {rar_path}...")
    # unrar should be pre-installed in standard Colab environments
    subprocess.run(["unrar", "x", "-o+", rar_path, DATA_DIR], check=True)

In [ ]:
# @title 1f. Verify dataset structure
print("Verifying structure...")
for expected in ["Cashew/Cashew-Uganda/images", "Coffee/Batch1/images"]:
    found = False
    for root, dirs, files in os.walk(DATA_DIR):
        if expected in root:
            found = True
            print(f"✓ Found {expected} at {root}")
            break
    if not found:
        print(f"Warning: Could not find expected directory structure containing '{expected}'")

In [ ]:
# @title 1g. Flatten Coffee & Convert YOLO → COCO JSON ground truth
import shutil
import sys

# Flatten coffee batches
coffee_flat_dir = os.path.join(DATA_DIR, "Coffee_flattened")
if os.path.exists(coffee_flat_dir):
    shutil.rmtree(coffee_flat_dir)

print("Flattening Coffee dataset...")
subprocess.run([sys.executable, f"{REPO_DIR}/common/flatten_coffee.py"], check=True)

# Convert to standard COCO splits
os.makedirs(os.path.join(DATA_DIR, "coco"), exist_ok=True)

# Cashew conversion
subprocess.run([
    sys.executable,
    f"{REPO_DIR}/common/yolo_to_coco.py",
    "--images", f"{DATA_DIR}/Cashew/Cashew-Uganda/images",
    "--labels", f"{DATA_DIR}/Cashew/Cashew-Uganda/Labels",
    "--class_map", f"{REPO_DIR}/common/class_map.json",
    "--dataset", "cashew",
    "--output", f"{DATA_DIR}/coco/",
    "--split_ratio", "0.8"
], check=True)

# Coffee conversion
subprocess.run([
    sys.executable,
    f"{REPO_DIR}/common/yolo_to_coco.py",
    "--images", f"{DATA_DIR}/Coffee_flattened/images",
    "--labels", f"{DATA_DIR}/Coffee_flattened/labels",
    "--class_map", f"{REPO_DIR}/common/class_map.json",
    "--dataset", "coffee",
    "--output", f"{DATA_DIR}/coco/",
    "--split_ratio", "0.8"
], check=True)

print("\n✓ Ground-truth COCO JSON splits created successfully.")

In [ ]:
# @title 1g-fix. Drop `tree` annotations + remap classes (scale-imbalance fix) {display-mode: "form"}
# `tree` boxes are whole-image scale (median 14% x 19% of the frame; many at 100% x 100%),
# ~40-60x the area of a fruit box. Keeping them (a) inflates macro AP@50 via trivially-easy
# large-box IoU and (b) competes with the tiny fruit boxes in the loss. We remove them at
# the single source of truth -- the generated COCO train/val JSONs -- BEFORE the COCO->YOLO
# step, then remap the surviving class ids to a contiguous 0..K-1 space so the YOLO labels,
# data.yaml, model predictions, and COCOeval all stay aligned. Drop is by class *name*, so
# it covers cashew ("tree") and coffee ("coffee_tree") uniformly and is idempotent.
import glob
import json
import os

DROP_CLASS_NAMES = ["tree", "coffee_tree"]  # @param {type:"raw"}


def drop_classes_and_remap(coco_path, drop_names):
    # Filters out annotations of dropped classes and remaps remaining category ids
    # to a contiguous 0..K-1 space. Rewrites the JSON in place. Returns the new names.
    drop_names = set(drop_names)
    with open(coco_path) as f:
        data = json.load(f)

    survivors = [c for c in data["categories"] if c["name"] not in drop_names]
    old2new = {c["id"]: i for i, c in enumerate(survivors)}
    dropped_ids = {c["id"] for c in data["categories"] if c["name"] in drop_names}

    data["categories"] = [{**c, "id": old2new[c["id"]]} for c in survivors]
    kept = []
    for a in data["annotations"]:
        if a["category_id"] in dropped_ids:
            continue
        kept.append({**a, "category_id": old2new[a["category_id"]]})

    n_before, n_after = len(data["annotations"]), len(kept)
    data["annotations"] = kept
    with open(coco_path, "w") as f:
        json.dump(data, f, indent=2)

    names = [c["name"] for c in data["categories"]]
    print(
        f"{os.path.basename(coco_path)}: dropped {n_before - n_after} annotations "
        f"({n_before} -> {n_after}); classes now {names}"
    )
    return names


coco_jsons = sorted(glob.glob("data/coco/*.json"))
if not coco_jsons:
    print("No COCO JSONs found in data/coco/ -- run cell 1g first.")
for p in coco_jsons:
    drop_classes_and_remap(p, DROP_CLASS_NAMES)

print(
    "\nDone. Now run Section 2 (COCO->YOLO) so YOLO labels + data.yaml adopt the new "
    "class space, then train. The ground-truth visualization below will also show no tree boxes."
)

### 1h. Visualize Ground Truth Bounding Boxes

Let's use the shared utility `common/visualize_coco.py` to draw ground truth bounding boxes on a few validation images to inspect the annotation density and categories before we restructure the data.

In [ ]:
# @title 1h. Visualize Ground Truth Annotations
import sys
sys.path.append(REPO_DIR)

from common.visualize_coco import visualize

# Visualize 3 random ground truth validation images for Cashew
print("Visualizing cashew validation ground truth annotations:")
visualize(
    coco_json="data/coco/cashew_val.json",
    image_dir="data/Cashew/Cashew-Uganda/images",
    num_images=3,
    output_dir=None
)

## 2. Dataset Restructuring (COCO → YOLO Split)

In [ ]:
# @title Configure & Run Conversion {display-mode: "form"}

DATASET = "cashew"  # @param ["cashew", "coffee"]
YOLO_DATA_DIR = f"data/yolo/{DATASET}"

COCO_TRAIN_JSON = f"data/coco/{DATASET}_train.json"
COCO_VAL_JSON = f"data/coco/{DATASET}_val.json"

IMAGE_DIRS = {
    "cashew": "data/Cashew/Cashew-Uganda/images",
    "coffee": "data/Coffee_flattened/images"
}
SRC_IMAGE_DIR = IMAGE_DIRS[DATASET]

print(f"Restructuring {DATASET} images to match COCO train/val splits...")

# Run our group's custom mapping script
subprocess.run([
    "python3",
    f"{REPO_DIR}/models/yolo26/scripts/coco_to_yolo.py",
    "--coco_train", COCO_TRAIN_JSON,
    "--coco_val", COCO_VAL_JSON,
    "--image_dir", SRC_IMAGE_DIR,
    "--output_dir", YOLO_DATA_DIR
], check=True)

print("\n✓ YOLO formatted directories and data.yaml prepared!")

## 3. Supervised YOLO26 Training

In [ ]:
# @title 3a. Enable Focal Loss for classification (imbalance fix) {display-mode: "form"}
# The Cashew set is class-imbalanced: spoilt/flower/premature dominate while the
# maturity classes that matter for yield (ripe ~9%, unripe ~6%) are scarce.
# Ultralytics exposes NO flag for focal loss and dropped YOLOv5's `fl_gamma`, so we
# swap the hardcoded classification BCE inside `v8DetectionLoss` for a focal-modulated
# drop-in. This must run BEFORE model.train(). It is version-safe: it patches the base
# v8DetectionLoss class, which YOLO26/11 and any end-to-end (E2E) wrapper instantiate
# internally. Toggle USE_FOCAL_LOSS off for a clean baseline ablation.

import torch
import torch.nn as nn
import torch.nn.functional as F

USE_FOCAL_LOSS = True   # @param {type:"boolean"}
FOCAL_GAMMA = 1.5       # @param {type:"number"}
FOCAL_ALPHA = 0.25      # @param {type:"number"}


class FocalLossBCE(nn.Module):
    """Drop-in replacement for nn.BCEWithLogitsLoss(reduction='none') with focal modulation.

    Returns an element-wise loss tensor with the SAME shape as the input, so the
    normalization done downstream in v8DetectionLoss (`.sum() / target_scores_sum`)
    is preserved exactly. Works with the soft alignment targets produced by YOLO's
    TaskAlignedAssigner (targets in [0, 1]), not just hard 0/1 labels.

    gamma: focusing parameter. Higher -> more weight on hard/misclassified boxes.
    alpha: positive/negative balancing in [0, 1]; set < 0 to disable alpha balancing.
    """

    def __init__(self, gamma=1.5, alpha=0.25):
        super().__init__()
        self.gamma = gamma
        self.alpha = alpha

    def forward(self, pred, target):
        bce = F.binary_cross_entropy_with_logits(pred, target, reduction="none")
        prob = pred.sigmoid()
        p_t = target * prob + (1 - target) * (1 - prob)
        loss = bce * (1.0 - p_t) ** self.gamma
        if self.alpha >= 0:
            alpha_t = target * self.alpha + (1 - target) * (1 - self.alpha)
            loss = loss * alpha_t
        return loss  # reduction='none' -> same shape as `pred`


from ultralytics.utils import loss as _ul_loss

# Cache the stock __init__ once so re-running this cell (or toggling off) restores cleanly.
if not hasattr(_ul_loss.v8DetectionLoss, "_orig_init"):
    _ul_loss.v8DetectionLoss._orig_init = _ul_loss.v8DetectionLoss.__init__


def _focal_init(self, *args, **kwargs):
    _ul_loss.v8DetectionLoss._orig_init(self, *args, **kwargs)
    self.bce = FocalLossBCE(gamma=FOCAL_GAMMA, alpha=FOCAL_ALPHA)


if USE_FOCAL_LOSS:
    _ul_loss.v8DetectionLoss.__init__ = _focal_init
    print(
        f"✓ Focal loss ENABLED (gamma={FOCAL_GAMMA}, alpha={FOCAL_ALPHA}). "
        "Classification BCE replaced with FocalLossBCE."
    )
else:
    _ul_loss.v8DetectionLoss.__init__ = _ul_loss.v8DetectionLoss._orig_init
    print("Focal loss DISABLED — using stock BCE classification loss.")

# NOTE: single-process training only (Colab T4 is fine). Under multi-GPU DDP, worker
# subprocesses re-import ultralytics and lose this patch; wire it via a callback instead.

In [ ]:
# @title 3b. Weighted sampling for class imbalance (imbalance fix) {display-mode: "form"}
# Oversample images containing rare classes (unripe ~6%, ripe ~9%) so the model sees them
# more often each epoch. Ultralytics dropped YOLOv5's `image-weights`, so we inject a
# WeightedRandomSampler into the TRAINING dataloader by patching build_dataloader. Each
# image is weighted by its RAREST present class (inverse frequency) -- weighting by total
# box count instead would just favour dense images, not rare ones. Fully guarded: any
# mismatch falls back to the default uniform loader so training never breaks.
import sys
import numpy as np
import torch
from torch.utils.data import WeightedRandomSampler
from ultralytics.data import build as _ulb
import ultralytics.data as _uld

USE_WEIGHTED_SAMPLING = True  # @param {type:"boolean"}

if not hasattr(_ulb, "_orig_build_dataloader"):
    _ulb._orig_build_dataloader = _ulb.build_dataloader


def _compute_image_weights(dataset):
    # Returns (per-image weight array, per-class instance counts).
    labels = dataset.labels
    nc = 0
    for lab in labels:
        c = np.asarray(lab["cls"]).reshape(-1).astype(int)
        if c.size:
            nc = max(nc, int(c.max()) + 1)
    nc = max(nc, 1)
    counts = np.zeros(nc)
    present = np.zeros((len(labels), nc), dtype=bool)
    for i, lab in enumerate(labels):
        for cid in np.asarray(lab["cls"]).reshape(-1).astype(int):
            counts[cid] += 1
            present[i, cid] = True
    class_w = 1.0 / np.maximum(counts, 1)
    img_w = np.where(present, class_w[None, :], 0.0).max(1)   # weight by rarest present class
    pos = img_w[img_w > 0]
    img_w[img_w == 0] = (pos.min() * 0.5) if pos.size else 1.0  # label-less images: still sampled, lightly
    return img_w, counts


def _weighted_build_dataloader(dataset, batch, workers, shuffle=True, rank=-1, drop_last=False):
    # Only weight the single-process TRAINING loader; val/DDP fall through to default.
    if not (USE_WEIGHTED_SAMPLING and shuffle and rank == -1):
        return _ulb._orig_build_dataloader(dataset, batch, workers, shuffle=shuffle, rank=rank, drop_last=drop_last)
    try:
        img_w, counts = _compute_image_weights(dataset)
        sampler = WeightedRandomSampler(torch.as_tensor(img_w, dtype=torch.double),
                                        num_samples=len(dataset), replacement=True)
        base = _ulb._orig_build_dataloader(dataset, batch, workers, shuffle=False, rank=rank, drop_last=drop_last)
        loader = type(base)(
            dataset=dataset,
            batch_size=base.batch_size,
            num_workers=base.num_workers,
            sampler=sampler,
            pin_memory=getattr(base, "pin_memory", True),
            collate_fn=base.collate_fn,
            worker_init_fn=getattr(base, "worker_init_fn", None),
            generator=getattr(base, "generator", None),
            drop_last=getattr(base, "drop_last", False),
        )
        ratio = img_w.max() / img_w[img_w > 0].min()
        print(f"✓ Weighted sampling ENABLED. Per-class instance counts {counts.astype(int).tolist()}. "
              f"Rarest-class images sampled up to ~{ratio:.1f}x more often.")
        return loader
    except Exception as e:
        print(f"⚠ Weighted sampling failed ({e}); using default uniform loader.")
        return _ulb._orig_build_dataloader(dataset, batch, workers, shuffle=shuffle, rank=rank, drop_last=drop_last)


def _install(fn):
    # Patch the source module, the package re-export (for late `from ultralytics.data
    # import build_dataloader`), and any module that already bound the symbol.
    _ulb.build_dataloader = fn
    _uld.build_dataloader = fn
    for _mod in list(sys.modules.values()):
        if _mod is None:
            continue
        if getattr(_mod, "build_dataloader", None) in (_ulb._orig_build_dataloader, fn):
            try:
                _mod.build_dataloader = fn
            except Exception:
                pass


if USE_WEIGHTED_SAMPLING:
    _install(_weighted_build_dataloader)
    print("Weighted-sampling patch installed (applies at next model.train()).")
else:
    _install(_ulb._orig_build_dataloader)
    print("Weighted sampling DISABLED — default uniform sampling.")

# NOTE: mosaic augmentation pulls its 3 extra tiles uniformly, so it dilutes the weighting
# a little; the primary (base) image of each sample is still drawn by the weighted sampler.

In [ ]:
# @title 3. Train model {display-mode: "form"}
from ultralytics import YOLO

EPOCHS = 50  # @param {type:"slider", min:5, max:100, step:5}
BATCH_SIZE = 16  # @param [8, 16, 32, 64]
IMAGE_SIZE = 640  # @param [320, 640, 960, 1280]

print("Initializing YOLO26 model...")
try:
    # Load lightweight Nano model for fast training on T4 GPU
    model = YOLO("yolo26n.pt")
except Exception:
    print("yolo26n.pt not found in Ultralytics hub. Falling back to yolo11n.pt...")
    model = YOLO("yolo11n.pt")

print("Starting supervised training...")
results = model.train(
    data=f"{YOLO_DATA_DIR}/data.yaml",
    epochs=EPOCHS,
    imgsz=IMAGE_SIZE,
    batch=BATCH_SIZE,
    device=0,
    project=f"runs/{DATASET}",
    name="train_run"
)

print("\n✓ Training complete!")

## 4. Validation Inference

In [ ]:
# @title 4. Run validation predictions {display-mode: "form"}
# COCO AP integrates over the FULL precision-recall curve, so detections must be
# generated at a very low confidence (~0.001). Filtering at a high conf here truncates
# recall and deflates AP -- and it hits SMALL objects hardest (their confidences are
# low), which is why the fruit classes scored ~0 while only the large tree/spoilt boxes
# survived. Keep any higher, human-facing threshold for the visualization cell only.
CONFIDENCE_THRESHOLD = 0.001  # @param {type:"number"}

# Use the BEST checkpoint, not the last-epoch weights left in `model` after training.
from ultralytics import YOLO
import os

_best = f"runs/{DATASET}/train_run/weights/best.pt"
if os.path.exists(_best):
    print(f"Loading best checkpoint: {_best}")
    model = YOLO(_best)
else:
    print("best.pt not found; using in-memory model (last-epoch weights).")

# Run inference on the validation split folder directly
val_images_dir = f"{YOLO_DATA_DIR}/images/val"

print(f"Running prediction inference on validation split: {val_images_dir}...")
pred_results = model.predict(
    source=val_images_dir,
    conf=CONFIDENCE_THRESHOLD,
    save=False,
    device=0,
)

print(f"\nInference complete. Generated predictions for {len(pred_results)} images.")

## 5. Parse Outputs to standard COCO format

In [ ]:
# @title 5. Format predictions to COCO JSON
import json  # required here: json.dump is used below but was only imported in Section 7
import sys
sys.path.append(REPO_DIR)

from models.yolo26.scripts.parse_output import yolo_results_to_coco
from pathlib import Path

predictions_output_json = f"runs/{DATASET}/yolo26_predictions.json"

print("Formatting results to COCO predictions JSON...")
coco_predictions = yolo_results_to_coco(pred_results, COCO_VAL_JSON)

# Ensure parent directories exist before writing
Path(predictions_output_json).parent.mkdir(parents=True, exist_ok=True)

# Save to predictions file
with open(predictions_output_json, "w") as f:
    json.dump(coco_predictions, f, indent=2)

print(f"Saved predictions to {predictions_output_json}")

## 6. Unified Evaluation (COCOEval)

In [ ]:
# @title 6. Run Unified Evaluation
import sys
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

# Append repository path to access common modules
if 'REPO_DIR' in globals() and REPO_DIR not in sys.path:
    sys.path.append(REPO_DIR)

from common.evaluate import evaluate_coco

# score_threshold=None: AP must integrate over ALL detections. Filtering here (as the
# template did at 0.25) truncates the precision-recall curve and collapses small-object
# AP to ~0. Report AP on the full set; apply an operating threshold only for deployment.
print("Loading validation ground truth and formatted predictions...")
results = evaluate_coco(
    gt_json=COCO_VAL_JSON,
    predictions=predictions_output_json,
    score_threshold=None,
    verbose=True
)

## 7. Visualise predictions vs ground truth

In [ ]:
# @title 7. Draw bounding boxes side-by-side
import json
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
import random

VIS_THRESHOLD = 0.3  # @param {type:"number"}
NUM_VIS = 3  # @param {type:"slider", min:1, max:6, step:1}

with open(COCO_VAL_JSON, "r") as f:
    coco_data = json.load(f)

categories = {cat["id"]: cat["name"] for cat in coco_data["categories"]}
images_list = coco_data["images"]

# Group GT annotations
gt_by_image = {}
for ann in coco_data["annotations"]:
    img_id = ann["image_id"]
    if img_id not in gt_by_image:
        gt_by_image[img_id] = []
    gt_by_image[img_id].append(ann)

# Group predictions
pred_by_image = {}
for pred in coco_predictions:
    if pred["score"] < VIS_THRESHOLD:
        continue
    img_id = pred["image_id"]
    if img_id not in pred_by_image:
        pred_by_image[img_id] = []
    pred_by_image[img_id].append(pred)

cmap = plt.cm.get_cmap("tab10")
cat_colors = {cat_id: cmap(i % 10) for i, cat_id in enumerate(categories.keys())}

def draw_boxes(ax, boxes, is_gt=True):
    for item in boxes:
        bbox = item["bbox"]
        cat_id = item["category_id"]
        color = cat_colors.get(cat_id, "red")
        linestyle = "-" if is_gt else "--"
        linewidth = 2 if is_gt else 1.5
        
        rect = patches.Rectangle(
            (bbox[0], bbox[1]), bbox[2], bbox[3],
            linewidth=linewidth, edgecolor=color,
            facecolor="none", linestyle=linestyle
        )
        ax.add_patch(rect)
        
        label = categories.get(cat_id, f"cls{cat_id}")
        if not is_gt and "score" in item:
            label = f"{label} {item['score']:.2f}"
        
        ax.text(
            bbox[0], bbox[1] - 4, label,
            fontsize=7, color=color,
            bbox=dict(boxstyle="round,pad=0.15", facecolor="black", alpha=0.6)
        )

# Randomly select subset of validation images
sample_images = random.sample(images_list, min(NUM_VIS, len(images_list)))

fig, axes = plt.subplots(len(sample_images), 2, figsize=(16, 5 * len(sample_images)))
if len(sample_images) == 1:
    axes = axes.reshape(1, -1)

for row, img_info in enumerate(sample_images):
    img_id = img_info["id"]
    img_path = Path(SRC_IMAGE_DIR) / img_info["file_name"]
    if not img_path.exists():
        img_path = Path(SRC_IMAGE_DIR) / img_info["file_name"].strip()
        
    image = Image.open(img_path).convert("RGB")
    
    # GT Bboxes
    axes[row, 0].imshow(image)
    axes[row, 0].set_title(f"Ground Truth — {img_info['file_name']}", fontsize=10)
    axes[row, 0].axis("off")
    draw_boxes(axes[row, 0], gt_by_image.get(img_id, []), is_gt=True)
    
    # Prediction Bboxes
    axes[row, 1].imshow(image)
    axes[row, 1].set_title(f"YOLO26 Predictions (vis_threshold={VIS_THRESHOLD})", fontsize=10)
    axes[row, 1].axis("off")
    draw_boxes(axes[row, 1], pred_by_image.get(img_id, []), is_gt=False)

plt.tight_layout()
plt.show()